# Human-in-the-Loop

*Notebook 22*

Pause agent execution at critical decision points and require human approval before continuing.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

# Notebook-specific imports
import json
import re


def flag_unsupported(text):
    """Heuristic scan for services/timelines these demo tools cannot do.
    A clean result is NOT proof the response is grounded, only that these
    known phrases were absent."""
    low = text.lower()
    terms = ["business day", "return label", "replacement", "slack", "teams",
             "1-800", "phone number", "refund will arrive", "3-7", "5-10",
             "original payment method", "credited to", "refunded to", "bank",
             "statement", "days"]
    hits = [t for t in terms if t in low]
    if re.search(r"\+?\d[\d\-\s]{6,}\d", text):   # phone-like number
        hits.append("phone-like number")
    return hits

MODEL = "gpt-5-mini"


print("✅ Ready!")

---

## 🎯 The Problem

Some agent actions are reversible: getting the weather, looking up a product.

For high-stakes actions, you want a human to review and approve before the tool executes.

---

## 🔄 Part 1: How HITL Works

An approval-required tool call does not execute until you approve it.

The approval flow:

1. `Runner.run()` returns when the model requests a protected tool.
2. `result.interruptions` contains the pending call. The tool has not run.
3. Convert the result with `state = result.to_state()`.
4. Approve or reject each interruption.
5. Resume with `Runner.run(agent, state)`.

Approved calls execute.

Rejected calls return a model-visible rejection result.

---

## ✉️ Part 2: Basic Approval Flow

We'll build an email agent where sending always requires human approval.

This matters because approval now happens in the SDK.

The tool cannot run just because the agent ignored its instructions.

In [ ]:
# Simulated send ledger: records (to, subject) so we can prove exactly what
# ran, and only after approval. No email actually leaves this notebook.
SENT_EMAILS = []

@function_tool
def draft_email(to: str, subject: str, body: str) -> str:
    """Draft an email (no approval needed, just composing)."""
    return f"Draft ready: To={to}, Subject={subject}, Body={body[:50]}..."


@function_tool(needs_approval=True)
def send_email(to: str, subject: str, body: str) -> str:
    """Simulate sending an email. Requires approval before it records a send."""
    SENT_EMAILS.append((to, subject))  # simulated side effect, only after approval
    return f"✅ Simulated send recorded (demo, no real email): {to}: {subject}"


instructions = (
    "Help users compose and send emails.\n"
    "Use draft_email to prepare the message, then send_email to record a simulated send.\n"
    "Only use these tools. Do not claim to save drafts, escalate, contact anyone,\n"
    "or offer channels or phone numbers. State only what the tools did."
)

email_agent = Agent(
    name="EmailAssistant",
    instructions=instructions,
    model=MODEL,
    tools=[draft_email, send_email]
)

# --------------------------------------------------------------
print("✅ Email agent ready")
print("   send_email requires approval, draft_email does not")

#### Step 1: Run Until Paused

In [ ]:
print("📧 EMAIL APPROVAL DEMO")
print("=" * 60)

SENT_EMAILS.clear()
result = await Runner.run(email_agent, input="Send an email to alex@example.com with subject 'Meeting Tomorrow' saying we'll meet at 2pm.")

# The demo requires the approval pause: fail loudly if it never happened
if not result.interruptions:
    print(result.final_output)
    raise RuntimeError("Expected send_email to pause for approval, but the run completed.")

print(f"⏸️  Run paused: {len(result.interruptions)} approval(s) pending\n")
for interruption in result.interruptions:
    print(f"🔔 Tool: {interruption.raw_item.name}")
    print(f"    Args: {interruption.raw_item.arguments}")
    # Note: arguments is a JSON string, use json.loads() to inspect specific fields

# Deterministic invariant: a paused approval-required tool has NOT run
assert not SENT_EMAILS, "send_email ran before approval: impossible with needs_approval."
print(f"\n    send_email has NOT run yet: {len(SENT_EMAILS)} sent")

Each interruption wraps a `raw_item`: the pending tool call.

Inspect its `.name` and `.arguments` (a JSON string).

#### Step 2: Approve and Resume

In [ ]:
if result.interruptions:
    # Capture the resumable state
    state = result.to_state()

    # This basic demo expects a single send to approve
    pending = state.get_interruptions()
    if len(pending) != 1:
        raise RuntimeError(f"Expected exactly one pending approval, got {len(pending)}.")

    # Validate BEFORE approving (Part 4 generalizes this). Here: only the
    # intended recipient is allowed; anything else is rejected, unexecuted.
    interruption = pending[0]
    call = interruption.raw_item
    args = json.loads(call.arguments)
    approved = call.name == "send_email" and args.get("to") == "alex@example.com"
    if approved:
        print(f"✅ Approving: {call.name} to {args.get('to')}")
        state.approve(interruption)
    else:
        print(f"🚫 Rejecting unexpected call: {call.name} to {args.get('to')}")
        state.reject(interruption, rejection_message="Only the intended email to alex@example.com is approved.")

    resumed_result = await Runner.run(email_agent, state)

    if resumed_result.interruptions:
        print("\n🔁 The model requested another approval (a second decision would be needed).")
    else:
        print(f"\n📨 Run resumed and completed")
        print(f"Response: {resumed_result.final_output}")
        hits = flag_unsupported(resumed_result.final_output)
        if hits:
            print(f"⚠️ Heuristic flag: response may mention things the tools don't do: {hits}")

    print(f"Simulated sends recorded: {len(SENT_EMAILS)}")
    # If the intended send was approved and completed, it ran exactly once
    if approved and not resumed_result.interruptions:
        assert len(SENT_EMAILS) == 1 and SENT_EMAILS[0][0] == "alex@example.com", f"Unexpected ledger: {SENT_EMAILS}"
    print("=" * 60)

    result = resumed_result   # prevent a re-run of this cell from replaying the original approval

### 💡 Key Takeaway

`needs_approval=True` marks a tool as requiring an approval decision: a human, or your code standing in for one.

The SDK pauses the run and surfaces the pending call in `result.interruptions`. It does not check who approves: these demos approve programmatically.

The tool never runs until you call `state.approve()`.

---

## ❌ Part 3: Rejecting a Tool Call

When you reject a pending call, the SDK returns a rejection result to the model and resumes.

The model may adapt, finish without the tool, or request another call.

In [ ]:
print("🚫 REJECTION DEMO")
print("=" * 60)

SENT_EMAILS.clear()
result = await Runner.run(email_agent, input="Send an urgent email to all-company@example.com saying the servers are down.")

if not result.interruptions:
    print(result.final_output)
    raise RuntimeError("Expected send_email to pause for approval, but the run completed.")

state = result.to_state()

for interruption in state.get_interruptions():
    print(f"🔔 Tool wants to run: {interruption.raw_item.name}")
    print(f"    Args: {interruption.raw_item.arguments}\n")

    # Reject, provide a reason the agent can use
    print("🚫 Rejecting: all-company emails require manager sign-off")
    state.reject(
        interruption,
        rejection_message="All-company emails require manager approval. Please save as draft instead."
    )

# Resume after recording the rejection

resumed_result = await Runner.run(email_agent, state)

if resumed_result.interruptions:
    print("\n🔁 The model requested another approval (rejection does not guarantee adaptation).")
else:
    print("\nAgent response after rejection:")
    print(resumed_result.final_output)
    hits = flag_unsupported(resumed_result.final_output)
    if hits:
        print(f"⚠️ Heuristic flag: response may mention things the tools don't do: {hits}")

# Deterministic invariant: a rejected call cannot run
assert not SENT_EMAILS, "A rejected send executed: impossible."
print(f"Simulated sends recorded: {len(SENT_EMAILS)} (rejected, so none ran)")
print("=" * 60)

A custom rejection message gives the model context for its next turn.

It does not guarantee adaptation or prevent another approval request.

---

## 🎚️ Part 4: Threshold-Based Escalation

Not every action needs the same level of oversight.

Auto-approve low-risk actions and escalate high-risk ones to a human, based on validated parameters.

The SDK does not choose the threshold for you.

You write the rule in Python, such as `if amount > 50: require approval`.

(`needs_approval` can also be a callable, so low-risk actions skip the pause entirely.)

In [ ]:
# Authoritative order data (the source of truth, not the model's arguments)
ORDERS = {
    "ORD-001": {"item": "Phone case", "price": 15.99},
    "ORD-002": {"item": "Headphones", "price": 349.00},
    "ORD-003": {"item": "Phone screen protector", "price": 8.50},
}

# Records simulated refunds so we can prove when the tool actually ran
PROCESSED_REFUNDS = []

@function_tool(needs_approval=True)
def process_refund(order_id: str, amount: float, reason: str) -> str:
    """Simulate processing a refund (demo, no real payment). Requires approval."""
    # Tool-level idempotency backup: never refund the same order twice
    if any(existing_id == order_id for existing_id, _ in PROCESSED_REFUNDS):
        return f"Refund already processed for order {order_id}"
    PROCESSED_REFUNDS.append((order_id, amount))
    return f"✅ Simulated refund recorded (demo, no real payment): ${amount:.2f} for order {order_id}"


@function_tool
def lookup_order(order_id: str) -> str:
    """Look up order details by ID."""
    order = ORDERS.get(order_id)
    if not order:
        return f"Order {order_id} not found"
    return f"${order['price']:.2f}: {order['item']}"


support_agent = Agent(
    name="SupportAgent",
    instructions=(
        "You are a customer support agent. Help customers with refunds and order issues.\n"
        "Use only lookup_order and process_refund. Do not invent refund timelines,\n"
        "payment behavior, replacements, or return labels. State only what the tools returned."
    ),
    model=MODEL,
    tools=[lookup_order, process_refund]
)

# --------------------------------------------------------------
print("✅ Support agent ready")

#### Threshold-Based Approval Logic

In [ ]:
async def handle_refund_request(customer_message, auto_approve_threshold=50.0):
    """Handle refund with fail-closed, threshold-based approval."""

    print(f"📞 Customer: {customer_message}")
    print("=" * 60)

    result = await Runner.run(support_agent, input=customer_message)

    # A while loop: the agent may call another approval-required tool after resuming
    while result.interruptions:
        state = result.to_state()

        for interruption in state.get_interruptions():
            tool_name = interruption.raw_item.name

            # Fail closed: reject a tool this gate does not recognize
            if tool_name != "process_refund":
                print(f"🚫 Rejecting unrecognized approval tool: {tool_name}")
                state.reject(interruption, rejection_message="This action is not permitted by the refund policy.")
                continue

            # Fail closed: reject arguments we cannot even parse
            try:
                args = json.loads(interruption.raw_item.arguments)
            except (json.JSONDecodeError, TypeError):
                print("🚫 Rejecting: refund arguments could not be parsed")
                state.reject(interruption, rejection_message="The request could not be read and was not approved.")
                continue

            if not isinstance(args, dict):
                print("🚫 Rejecting: refund arguments were not a valid object")
                state.reject(interruption, rejection_message="Refund arguments were not a valid object.")
                continue

            order_id = args.get("order_id")
            amount = args.get("amount")
            reason = args.get("reason")
            order = ORDERS.get(order_id) if isinstance(order_id, str) else None

            # Fail closed: reject anything that does not match the source of truth.
            # Exact match for these two-decimal fixtures; production money should
            # use integer cents or Decimal, not float compares.
            if (order is None or not isinstance(amount, (int, float)) or isinstance(amount, bool)
                    or amount != order["price"]):
                print(f"🚫 Rejecting: refund could not be validated (order={order_id}, amount={amount})")
                state.reject(interruption, rejection_message="Refund details could not be validated against order records.")
                continue

            # Fail closed: require a nonempty reason (untrusted customer/model text)
            if not isinstance(reason, str) or not reason.strip():
                print("🚫 Rejecting: refund is missing a reason")
                state.reject(interruption, rejection_message="A refund reason is required.")
                continue

            # Fail closed: never refund the same order twice
            if any(existing_id == order_id for existing_id, _ in PROCESSED_REFUNDS):
                print(f"🚫 Rejecting: order {order_id} was already refunded")
                state.reject(interruption, rejection_message="This order was already refunded.")
                continue

            # From here the refund is validated against trusted data
            if amount <= auto_approve_threshold:
                print(f"✅ Auto-approved: ${amount:.2f} refund (validated, below ${auto_approve_threshold} threshold)")
                state.approve(interruption)
            else:
                print(f"⚠️  Manual review required: ${amount:.2f} refund (validated, above threshold)")
                print(f"    Order: {order_id} ({order['item']})")
                print(f"    Reason (untrusted): {reason}")
                # In your own app, replace this with a real approval source:
                # a UI button, a Slack reply, or a CLI input() prompt.
                decision = "yes"  # Simulated reviewer decision for this demo
                if decision in ["yes", "y"]:
                    print("    ✅ Simulated reviewer approved")
                    state.approve(interruption)
                else:
                    print("    🚫 Simulated reviewer rejected")
                    state.reject(interruption, rejection_message="Refund requires additional verification. Please contact billing.")

        result = await Runner.run(support_agent, state)

    print(f"\n💬 Agent: {result.final_output}")
    return result


# --------------------------------------------------------------
print("✅ handle_refund_request() ready")

### 💡 Key Takeaway

The same `needs_approval=True` flag serves both paths: auto-approve and human escalation.

The SDK doesn't distinguish between them: it always pauses and surfaces the interruption.

⚠️ **Security note:** Refunds and other financial writes have high-impact side effects.

Validate them against trusted application data.

Require approval above the policy threshold, and never rely on the model's judgment alone.

This demo validates order existence and price only. Production must also verify authenticated ownership, refund eligibility and status, and durable audit state.

#### Test: Small Refund (Auto-Approved)

In [ ]:
# Small refund, auto-approved
PROCESSED_REFUNDS.clear()
res = await handle_refund_request(
    "Please process a full refund for order ORD-001. The phone case broke after one day.",
    auto_approve_threshold=50.0
)
processed = len(PROCESSED_REFUNDS)
print(f"\nRefunds recorded (simulated): {processed} (expected 1)")
if processed != 1:
    print("⚠️ The model did not process the validated refund this run (an arg mismatch fails closed, which is correct).")
hits = flag_unsupported(res.final_output)
if hits:
    print(f"⚠️ Heuristic flag: response may mention things the tools don't do: {hits}")

#### Test: Large Refund (Simulated Human Review)

The approval decision is hardcoded to `"yes"` for this demo.

In a real implementation, this would prompt the user via a UI or `input()` call before proceeding.

In [ ]:
# Large refund, requires manual review
PROCESSED_REFUNDS.clear()
res = await handle_refund_request(
    "Please process a full refund for order ORD-002. The headphones never worked properly.",
    auto_approve_threshold=50.0
)
processed = len(PROCESSED_REFUNDS)
print(f"\nRefunds recorded (simulated): {processed} (expected 1, after simulated review)")
if processed != 1:
    print("⚠️ The model did not process the validated refund this run (an arg mismatch fails closed, which is correct).")
hits = flag_unsupported(res.final_output)
if hits:
    print(f"⚠️ Heuristic flag: response may mention things the tools don't do: {hits}")

---

## 💪 Practice Exercises

### Exercise 1: File Operations Agent

*Covers: approval interruptions, read vs. delete permission tiers*

Build an agent with read (no approval) and delete (requires approval) file tools.

⚠️ **Security note:** Filesystem delete tools are high-risk side-effect tools.

In real systems, require strict scoping and explicit approval before any write or delete.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: File Operations Agent
# --------------------------------------------------------------
# Objective: Read requires no approval in this exercise.
#            Delete requires approval and strict path scoping.

# Simulated file store, so deletion is a real state change you can verify
FILES = {"config.json": "{ ...config... }", "temp.log": "debug output"}

# TODO 1: Create a read_file tool (no approval needed)
#           Returns FILES[filename], or a not-found message

# TODO 2: Create a delete_file tool with needs_approval=True
#           Removes filename from FILES and returns a confirmation

# TODO 3: Create a FileAgent with both tools

# TODO 4: Run with: "Read config.json and then delete temp.log"
#           Before approving: inspect the interruption's tool name and filename,
#           and only approve delete_file for "temp.log" (reject anything else).
#           Confirm "temp.log" is still in FILES before approval (not deleted yet),
#           then approve, resume, and confirm "temp.log" is gone. Print the output.

# --- Write your code below this line ---

### Exercise 2: Budget-Aware Purchase Agent

*Covers: approval interruptions, threshold-based auto-approval*

Build an agent that auto-approves small purchases and requires approval for large ones.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Budget-Aware Purchase Agent
# --------------------------------------------------------------
# Objective: Auto-approve purchases under $25, require approval above.
#            Validate prices against trusted catalog data, never the model's price.

# Trusted product catalog (the source of truth for prices)
CATALOG = {"pen": 3.99, "standing desk": 450.00}

# Records simulated purchases so you can prove when the tool actually ran
PURCHASES = []

# TODO 1: Create a make_purchase tool with needs_approval=True
#           Parameters: item (str), price (float)
#           Add a tool-level duplicate guard (don't purchase the same item
#           twice), mirroring the refund tool's backup
#           Appends to PURCHASES and returns a simulated-purchase confirmation

# TODO 2: Create a PurchaseAgent with this tool

# TODO 3: Write a purchase_handler() function that:
#           - Runs the agent
#           - If interrupted, parse the tool arguments
#           - Fail closed: reject unknown tools, malformed JSON, non-dict args,
#             non-numeric/boolean prices, unknown items, price != CATALOG[item],
#             and already-purchased items
#           - Auto-approves validated purchases <= 25
#           - Prints "Manual approval required" and approves (simulated reviewer)
#             validated purchases > 25
#           - Resumes and prints final_output

# TODO 4: Test and prove the side effect with len(PURCHASES):
#           - Invalid or mismatched purchase: 0 recorded
#           - Valid small purchase ("Buy a pen for $3.99"): exactly 1
#           - Large purchase ("Buy a standing desk for $450"): 0 before review, 1 after

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Mark a tool `needs_approval=True` to pause for sign-off:**

- `Runner.run()` pauses, inspect `result.interruptions`

- Call `state.approve(interruption)` or `state.reject(interruption, rejection_message="...")`

- Resume with `Runner.run(agent, state)`

- The tool never runs until you call `state.approve()`

- Rejection returns feedback the model may or may not act on
<br>
<br>

**Auto-approve low-risk actions, escalate only past a threshold:**

- Auto-approve only validated low-risk actions

- Escalate validated high-risk actions to humans

- Reject calls that cannot be validated

---

## 📍 Next Step

**Notebook 23: Tracing & Observability**  

Read traces in the OpenAI dashboard to inspect tool calls, handoffs, and debug bad runs.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-22-human-in-the-loop)